## Example 1: Basic Classification with XGBoost

In [ ]:
# Example 1: Basic Classification with XGBoost
import xgboost as xgb
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Generate synthetic classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_classes=2, 
                         n_informative=8, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train XGBoost classifier
model = xgb.XGBClassifier(
    objective='binary:logistic',
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

# Make predictions and evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"XGBoost version: {xgb.__version__}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test accuracy: {accuracy:.4f}")
print(f"Feature importance (top 3): {model.feature_importances_[:3]}")

## Example 2: Regression with Cross-Validation

In [ ]:
# Example 2: Regression with Cross-Validation
from sklearn.datasets import make_regression
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

# Generate regression dataset
X, y = make_regression(n_samples=1000, n_features=15, n_informative=10, 
                      noise=0.1, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to DMatrix for better performance
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

# Cross-validation with early stopping
params = {
    'objective': 'reg:squarederror',
    'max_depth': 6,
    'eta': 0.1,
    'eval_metric': 'rmse'
}

cv_results = xgb.cv(
    params, dtrain, num_boost_round=200, nfold=5,
    early_stopping_rounds=20, seed=42, verbose_eval=False
)

# Train final model
best_rounds = len(cv_results)
model = xgb.train(params, dtrain, num_boost_round=best_rounds)

# Make predictions
y_pred = model.predict(dtest)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Best rounds (CV): {best_rounds}")
print(f"Test MSE: {mse:.4f}")
print(f"Test R²: {r2:.4f}")
print(f"CV RMSE (final): {cv_results['test-rmse-mean'].iloc[-1]:.4f}")

## Example 3: Feature Selection and Importance Analysis

In [ ]:
# Example 3: Feature Selection and Importance Analysis
import matplotlib.pyplot as plt
import pandas as pd

# Create dataset with mixed relevance features
X, y = make_classification(n_samples=800, n_features=20, n_informative=8,
                         n_redundant=4, n_clusters_per_class=1, random_state=42)

# Add feature names
feature_names = [f'feature_{i}' for i in range(X.shape[1])]
X_df = pd.DataFrame(X, columns=feature_names)

X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.25, random_state=42)

# Train model with feature importance tracking
model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    importance_type='gain'
)

model.fit(X_train, y_train)

# Get feature importance scores
importance_scores = model.feature_importances_
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': importance_scores
}).sort_values('importance', ascending=False)

# Select top features
top_features = feature_importance.head(10)['feature'].tolist()
X_train_selected = X_train[top_features]
X_test_selected = X_test[top_features]

# Compare performance with selected features
model_selected = xgb.XGBClassifier(n_estimators=150, max_depth=4, 
                                 learning_rate=0.1, random_state=42)
model_selected.fit(X_train_selected, y_train)

# Evaluate both models
full_accuracy = accuracy_score(y_test, model.predict(X_test))
selected_accuracy = accuracy_score(y_test, model_selected.predict(X_test_selected))

print(f"Features used: {X.shape[1]} -> {len(top_features)}")
print(f"Full model accuracy: {full_accuracy:.4f}")
print(f"Selected features accuracy: {selected_accuracy:.4f}")
print(f"Top 5 important features: {top_features[:5]}")
print(f"Importance range: {importance_scores.min():.4f} - {importance_scores.max():.4f}")

## Example 4: Hyperparameter Tuning with GridSearch

In [ ]:
# Example 4: Hyperparameter Tuning with GridSearch
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

# Generate balanced dataset for tuning
X, y = make_classification(n_samples=600, n_features=12, n_classes=2,
                         n_informative=10, class_sep=0.8, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Define parameter grid for tuning
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 4, 6],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

# Create base model
xgb_model = xgb.XGBClassifier(random_state=42, eval_metric='logloss')

# Perform grid search with cross-validation
grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    scoring='accuracy',
    cv=3,
    n_jobs=-1,
    verbose=0
)

print("Starting hyperparameter tuning...")
grid_search.fit(X_train, y_train)

# Get best model and evaluate
best_model = grid_search.best_estimator_
best_predictions = best_model.predict(X_test)
best_accuracy = accuracy_score(y_test, best_predictions)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.4f}")
print(f"Test accuracy: {best_accuracy:.4f}")
print(f"Total combinations tested: {len(grid_search.cv_results_['params'])}")

# Show performance improvement
baseline_model = xgb.XGBClassifier(random_state=42)
baseline_model.fit(X_train, y_train)
baseline_accuracy = accuracy_score(y_test, baseline_model.predict(X_test))
improvement = best_accuracy - baseline_accuracy

print(f"Baseline accuracy: {baseline_accuracy:.4f}")
print(f"Improvement: {improvement:.4f} ({improvement/baseline_accuracy*100:.1f}%)")

## Example 5: Production Model Pipeline with Early Stopping

In [ ]:
# Example 5: Production Model Pipeline with Early Stopping
import time
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Create larger, more realistic dataset
X, y = make_classification(n_samples=2000, n_features=25, n_informative=15,
                         n_redundant=5, n_clusters_per_class=2, 
                         class_sep=0.7, random_state=42)

# Split data with validation set
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

# Create production-ready model with early stopping
model = xgb.XGBClassifier(
    objective='binary:logistic',
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=20,
    eval_metric='auc',
    random_state=42
)

# Train with validation monitoring
start_time = time.time()
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)
training_time = time.time() - start_time

# Evaluate final model
test_predictions = model.predict(X_test)
test_proba = model.predict_proba(X_test)[:, 1]
final_accuracy = accuracy_score(y_test, test_predictions)

# Model performance metrics
from sklearn.metrics import roc_auc_score, precision_score, recall_score

auc_score = roc_auc_score(y_test, test_proba)
precision = precision_score(y_test, test_predictions)
recall = recall_score(y_test, test_predictions)

print(f"\nModel Performance Summary:")
print(f"Training time: {training_time:.2f} seconds")
print(f"Stopped at iteration: {model.best_iteration}")
print(f"Test accuracy: {final_accuracy:.4f}")
print(f"AUC score: {auc_score:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Best validation score: {model.best_score:.4f}")

# Feature importance for interpretability
top_5_features = np.argsort(model.feature_importances_)[-5:]
print(f"Top 5 feature indices: {top_5_features}")
print("Production model ready for deployment!")

## Example 6: Distributed training with xgboost.dask

In [ ]:
import numpy as np
import dask.array as da
import xgboost as xgb
import xgboost.dask
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=2, threads_per_worker=1, processes=False, dashboard_address=":0")
client = Client(cluster)
try:
    rng = np.random.default_rng(0)
    features = rng.normal(size=(512, 4))
    labels = (features[:, 0] + features[:, 1] > 0).astype(int)
    dtrain = xgb.dask.DaskDMatrix(client, da.from_array(features, chunks=(256, 4)),
                                  da.from_array(labels, chunks=(256,)))
    assert dtrain.num_col() == 4, dtrain.num_col()  # DaskDMatrix read the column count from the partitions
    trained = xgb.dask.train(client, {"objective": "binary:logistic", "tree_method": "hist",
                                      "max_depth": 3}, dtrain, num_boost_round=8)
    booster = trained["booster"]
    assert len(booster.get_dump()) == 8, len(booster.get_dump())   # every round produced a tree
    predicted = xgb.dask.predict(client, booster, dtrain).compute()
    accuracy = float(((predicted > 0.5).astype(int) == labels).mean())
    assert accuracy > 0.9, accuracy   # a lost partition would lower the accuracy
    print(f"xgboost.dask: 2 partitions, 8 rounds, accuracy {accuracy:.3f}")
finally:
    client.close()
    cluster.close()

## Example 7: GPU training (device="cuda")

In [ ]:
import json
import numpy as np
import xgboost as xgb

info = xgb.build_info()
assert info["USE_CUDA"], f"requires-gpu cell ran but this libxgboost was built USE_CUDA=OFF: {info['libxgboost']}"
rng = np.random.default_rng(7)
X = rng.random((20_000, 16), dtype=np.float32)
y = (3 * X[:, 0] + X[:, 1] ** 2 + rng.normal(0, 0.05, 20_000)).astype(np.float32)
gpu = xgb.XGBRegressor(tree_method="hist", device="cuda", n_estimators=40, max_depth=6).fit(X, y)
config = json.loads(gpu.get_booster().save_config())  # the booster's saved config records which updater ran
assert "grow_gpu_hist" in json.dumps(config), "device='cuda' did not select the grow_gpu_hist updater"
cpu = xgb.XGBRegressor(tree_method="hist", device="cpu", n_estimators=40, max_depth=6).fit(X, y)
corr = float(np.corrcoef(gpu.predict(X[:4000]), cpu.predict(X[:4000]))[0, 1])
assert corr > 0.99, f"GPU and CPU boosters disagree on the same data/params: corr={corr:.4f}"
device = config.get("learner", {}).get("generic_param", {}).get("device", "?")
print(f"GPU: grow_gpu_hist on device={device}, gpu-vs-cpu prediction corr {corr:.4f}")

## Summary

This notebook demonstrated 7 comprehensive aspects of the **XGBoost** package:

1. **Basic Classification** - Standard binary classification with XGBoost classifier and performance evaluation
2. **Regression with Cross-Validation** - Regression tasks using DMatrix format with cross-validation and early stopping
3. **Feature Selection and Importance Analysis** - Feature importance scoring and selection for model optimization
4. **Hyperparameter Tuning with GridSearch** - Systematic hyperparameter optimization using GridSearchCV
5. **Production Model Pipeline** - Complete production-ready pipeline with early stopping and comprehensive metrics
6. **Distributed Training** - Training and prediction with `xgboost.dask` on a local Dask cluster
7. **GPU Training** - The `grow_gpu_hist` updater selected by `device="cuda"`, checked against a CPU booster

XGBoost provides efficient gradient boosting with excellent performance for both classification and regression tasks.